In [1]:
import pandas as pd
df = pd.read_csv("Dataset/Processed/cleaned_crime.csv", parse_dates=["Date"])
print(df.shape)

(8381931, 5)


In [2]:
year = 2023
df_y = df[df["Year"] == year].copy()

# Optional: focus on one crime type first to reduce density
# df_y = df_y[df_y["Primary Type"] == "THEFT"].copy()

print(df_y.shape)
df_y.describe(include="all")

(261243, 5)


,Date,Primary Type,Latitude,Longitude,Year
count,261243,261243,261243.000000,261243.000000,261243.0
unique,NaN,31,NaN,NaN,NaN
top,NaN,THEFT,NaN,NaN,NaN
freq,NaN,57194,NaN,NaN,NaN
mean,2023-07-04 19:45:01.839459840,NaN,41.846519,-87.668779,2023.0
min,2023-01-01 00:00:00,NaN,41.644590,-87.939733,2023.0
25%,2023-04-08 14:09:00,NaN,41.770847,-87.710219,2023.0
50%,2023-07-07 12:30:00,NaN,41.863582,-87.662214,2023.0
75%,2023-09-30 16:00:00,NaN,41.910007,-87.626819,2023.0
max,2023-12-31 23:59:00,NaN,42.022549,-87.524532,2023.0


In [3]:
df_y = df_y.sample(n=min(100000, len(df_y)), random_state=42)

In [4]:
import numpy as np
from sklearn.cluster import DBSCAN

# Convert lat/lon to radians for haversine distance
coords = np.radians(df_y[["Latitude", "Longitude"]].to_numpy())

# eps in radians: (meters / earth_radius_m)
earth_radius_m = 6371000
eps_m = 300  # try 200–500 to start
eps = eps_m / earth_radius_m

min_samples = 50  # try 30–150 depending on slice size

db = DBSCAN(eps=eps, min_samples=min_samples, metric="haversine", algorithm="ball_tree")
labels = db.fit_predict(coords)

df_y["cluster"] = labels

n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
noise_pct = (labels == -1).mean() * 100

print("Clusters:", n_clusters)
print("Noise %:", round(noise_pct, 2))
df_y["cluster"].value_counts().head(10)

Clusters: 63
Noise %: 10.23


cluster
 0     76304
-1     10232
 3      2697
 8      2519
 19      495
 14      494
 17      469
 12      349
 10      343
 23      325
Name: count, dtype: int64

In [5]:
from sklearn.metrics import silhouette_score

mask = df_y["cluster"] != -1
X = coords[mask]
labels = df_y.loc[mask, "cluster"]

sil = silhouette_score(X, labels, metric="euclidean")
print("Silhouette:", sil)

Silhouette: -0.5822355836253362


In [6]:
from sklearn.metrics import davies_bouldin_score

dbi = davies_bouldin_score(X, labels)
print("DB Index:", dbi)

DB Index: 1.0132852267775803


In [7]:
cluster_summary = (
    df_y[df_y["cluster"] != -1]
    .groupby("cluster")
    .agg(
        size=("cluster", "count"),
        mean_lat=("Latitude", "mean"),
        mean_lon=("Longitude", "mean")
    )
    .reset_index()
)

cluster_summary.to_csv("Outputs/Tables/spatial_cluster_summary_2023.csv",index=False)

print("Saved: spatial_cluster_summary_2023.csv")


Saved: spatial_cluster_summary_2023.csv
